# 🔧 Functions & Modules
---
## 📖 ReadMe
This notebook covers **defining functions**, advanced argument patterns, **lambda**, **decorators**, **closures**, and **modules/packages**.

| Section | Topics |
|---------|--------|
| Functions | def, return, scope, docstrings |
| Arguments | positional, keyword, *args, **kwargs, defaults |
| Advanced | lambda, closures, decorators, generators |
| Modules | import, from…import, __name__, packages |
| Standard Library | os, sys, math, datetime, random, itertools |

> **Level:** Intermediate  
> **Estimated Time:** 120 minutes


## 🧠 Concept Notes

### Function Anatomy
```python
def function_name(param1, param2=default, *args, **kwargs) -> return_type:
    """Docstring: what it does, args, returns."""
    # body
    return result
```

### Argument Types (order matters!)
```
def f(pos1, pos2, /, normal, *, kw_only, **kwargs)
       ↑ positional-only   ↑ keyword-only
```

### Scope — LEGB Rule
| Letter | Scope |
|--------|-------|
| **L** | Local — inside the function |
| **E** | Enclosing — outer function (closures) |
| **G** | Global — module level |
| **B** | Built-in — Python builtins |

### Decorators
A decorator is a **function that wraps another function** to add behaviour without modifying its source.

### Generators
Use `yield` instead of `return` — produces values **lazily** (one at a time), saving memory.


## 🖼️ Diagrams

In [ ]:
# Decorator call flow diagram
diagram = '''
┌──────────────────────────────────────────────┐
│              @decorator syntax                │
│                                              │
│   @timer                                     │
│   def slow_fn():  ←──── equivalent to ────► │
│       ...                 slow_fn = timer(slow_fn)
│                                              │
│   Call flow:                                 │
│   slow_fn()                                  │
│      │                                       │
│      ▼                                       │
│   wrapper()   ← added behaviour (before)    │
│      │                                       │
│      ▼                                       │
│   original slow_fn()                         │
│      │                                       │
│      ▼                                       │
│   wrapper()   ← added behaviour (after)     │
└──────────────────────────────────────────────┘
'''
print(diagram)

# LEGB scope diagram
scope_diagram = '''
Built-in (len, print, …)
  └── Global (module level)
        └── Enclosing (outer function — closures)
              └── Local (inside current function)
'''
print(scope_diagram)


## ✅ Executable Code

In [ ]:
# ── 1. Basic Functions ──
def greet(name: str, greeting: str = "Hello") -> str:
    """Return a personalised greeting.
    
    Args:
        name: The person's name.
        greeting: Greeting word (default 'Hello').
    Returns:
        Formatted greeting string.
    """
    return f"{greeting}, {name}!"

print(greet("Alice"))
print(greet("Bob", greeting="Hi"))
print(greet.__doc__)


In [ ]:
# ── 2. *args and **kwargs ──
def summarise(*args, **kwargs):
    print(f"Positional args: {args}")
    print(f"Keyword  args:   {kwargs}")
    return sum(args)

result = summarise(1, 2, 3, 4, label="total", precision=2)
print(f"Sum: {result}")

# Unpacking into calls
def add(a, b, c): return a + b + c
nums = [1, 2, 3]
opts = {"a": 10, "b": 20, "c": 30}
print(add(*nums))   # 6
print(add(**opts))  # 60


In [ ]:
# ── 3. Lambda & Higher-Order Functions ──
square = lambda x: x ** 2
print(square(5))

nums = [3, 1, 4, 1, 5, 9, 2, 6]
print("Sorted:", sorted(nums))
print("Sorted desc:", sorted(nums, reverse=True))

people = [{"name": "Alice", "age": 30}, {"name": "Bob", "age": 25}]
by_age = sorted(people, key=lambda p: p["age"])
print("By age:", by_age)

# map / filter
squares   = list(map(lambda x: x**2, range(1, 6)))
evens     = list(filter(lambda x: x % 2 == 0, range(10)))
print("Squares:", squares)
print("Evens:", evens)


In [ ]:
# ── 4. Closures ──
def make_multiplier(factor):
    """Returns a closure that multiplies by factor."""
    def multiplier(x):
        return x * factor       # 'factor' is captured from enclosing scope
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))   # 10  15

# Counter closure
def make_counter(start=0):
    count = [start]            # list so we can mutate it
    def increment(by=1):
        count[0] += by
        return count[0]
    def reset():
        count[0] = start
    increment.reset = reset
    return increment

counter = make_counter()
print(counter(), counter(), counter())   # 1 2 3
counter.reset()
print(counter())                         # 1


In [ ]:
# ── 5. Decorators ──
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[timer] {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

def retry(times=3, exceptions=(Exception,)):
    """Retry decorator — retry on failure up to `times` attempts."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"Attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
        return wrapper
    return decorator

@timer
def slow_sum(n):
    return sum(range(n))

@retry(times=3, exceptions=(ValueError,))
def risky(x):
    if x < 0:
        raise ValueError("negative!")
    return x * 2

print(slow_sum(1_000_000))
print(risky(5))


In [ ]:
# ── 6. Generators ──
def fibonacci(limit):
    """Yield Fibonacci numbers up to limit."""
    a, b = 0, 1
    while a <= limit:
        yield a
        a, b = b, a + b

print("Fibonacci ≤ 100:", list(fibonacci(100)))

# Generator expression (lazy)
gen = (x**2 for x in range(10**6))
print("First 5 squares:", [next(gen) for _ in range(5)])

# Infinite generator
def naturals(start=1):
    n = start
    while True:
        yield n
        n += 1

nat = naturals()
print("First 10 naturals:", [next(nat) for _ in range(10)])


In [ ]:
# ── 7. Modules & Imports ──
import math
import random
import datetime
from itertools import islice, chain, combinations

print(f"π = {math.pi:.6f}")
print(f"e = {math.e:.6f}")
print(f"sqrt(2) = {math.sqrt(2):.6f}")

random.seed(42)
print("Random ints:", [random.randint(1, 10) for _ in range(5)])
print("Random choice:", random.choice(["a", "b", "c", "d"]))

now = datetime.datetime.now()
print(f"Now: {now:%Y-%m-%d %H:%M:%S}")

# itertools examples
pairs = list(combinations("ABCD", 2))
print(f"Combinations of ABCD choose 2: {pairs}")
print("Chained:", list(chain([1,2], [3,4], [5])))


## 📝 Exercises

1. **Power Function** — Write `power(base, exp)` using recursion and without using `**`.
2. **Memoize Decorator** — Implement a `@memoize` decorator using a dict cache.
3. **Flatten Generator** — Write a generator `flatten(nested)` that yields all items from a nested list.
4. **Partial Application** — Implement `partial(func, *fixed_args)` from scratch (like `functools.partial`).
5. **Module Profiler** — Write a decorator that counts how many times each function is called.


## ✔️ Solutions

In [ ]:
# 1. Recursive power
def power(base, exp):
    if exp == 0: return 1
    if exp < 0:  return 1 / power(base, -exp)
    return base * power(base, exp - 1)

print(power(2, 10), power(3, -2))

# 2. Memoize decorator
def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    wrapper.cache = cache
    return wrapper

@memoize
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print([fib(i) for i in range(10)])

# 3. Flatten generator
def flatten(nested):
    for item in nested:
        if isinstance(item, (list, tuple)):
            yield from flatten(item)
        else:
            yield item

print(list(flatten([1, [2, [3, 4], 5], [6, 7]]]))

# 4. Partial application
def partial(func, *fixed_args, **fixed_kwargs):
    def wrapper(*args, **kwargs):
        return func(*fixed_args, *args, **{**fixed_kwargs, **kwargs})
    return wrapper

add5 = partial(lambda a, b: a + b, 5)
print(add5(3), add5(10))

# 5. Call counter decorator
def call_counter(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        wrapper.calls += 1
        return func(*a, **kw)
    wrapper.calls = 0
    return wrapper

@call_counter
def greet(name): return f"Hi {name}"

greet("A"); greet("B"); greet("C")
print(f"greet called {greet.calls} times")


## 💼 Enterprise Examples

In [ ]:
# Enterprise: Middleware-style decorator stack (like Flask/FastAPI)
import functools, time, logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        log.info(f"CALL  {func.__name__}  args={args}  kwargs={kwargs}")
        result = func(*args, **kwargs)
        log.info(f"RETURN {func.__name__} → {result}")
        return result
    return wrapper

def validate_positive(*param_indices):
    """Ensure specified positional params are positive numbers."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for idx in param_indices:
                if idx < len(args) and args[idx] <= 0:
                    raise ValueError(f"Param {idx} must be positive, got {args[idx]}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@log_call
@validate_positive(0, 1)
def compute_roi(investment: float, returns: float) -> float:
    """Return on Investment as a percentage."""
    return (returns - investment) / investment * 100

print(f"ROI: {compute_roi(1000.0, 1250.0):.1f}%")


## ❓ Interview Questions

**Q1: What is the difference between a function and a method?**
> A **method** is a function defined inside a class and called on an instance. A **function** is standalone.

**Q2: What are `*args` and `**kwargs`?**
> `*args` collects extra **positional** arguments into a tuple; `**kwargs` collects extra **keyword** arguments into a dict.

**Q3: What is a closure and when would you use one?**
> A closure is a function that remembers variables from its enclosing scope even after that scope has exited. Used for factory functions, decorators, and stateful callbacks.

**Q4: What does `functools.wraps` do?**
> It copies the wrapped function's `__name__`, `__doc__`, and `__module__` onto the wrapper, preserving introspection.

**Q5: Difference between `return` and `yield`?**
> `return` exits the function and discards state. `yield` suspends it, preserving local state — making it a **generator**.

**Q6: What is `__name__ == "__main__"`?**
> It checks whether the file is being **run directly** (not imported). Best practice for entry-point guards.


## ⚠️ Common Mistakes

| Mistake | Example | Fix |
|---------|---------|-----|
| Mutable default argument | `def f(lst=[])` | `def f(lst=None): lst = lst or []` |
| Forgetting `return` | `def add(a,b): a+b` | `return a + b` |
| Late binding in closures | `funcs = [lambda: i for i in range(3)]` all return 2 | `lambda i=i: i` |
| Shadowing builtins | `list = [1,2,3]` | Use different names |
| Not using `functools.wraps` | Decorator breaks `__name__` | Always add `@functools.wraps(func)` |
| Generator exhaustion | `g = gen(); list(g); list(g)` → `[]` | Recreate generator |


## 📄 Cheat Sheet
```python
# FUNCTION DEFINITION
def f(a, b=10, *args, kw_only, **kwargs): ...

# LAMBDA
fn = lambda x, y=0: x + y

# DECORATOR
def deco(func):
    @functools.wraps(func)
    def wrapper(*a, **kw):
        # before
        result = func(*a, **kw)
        # after
        return result
    return wrapper

# GENERATOR
def gen():
    yield value

# KEY BUILTINS
map(fn, iterable)
filter(fn, iterable)
sorted(it, key=fn, reverse=True)
functools.reduce(fn, it, initializer)
functools.partial(fn, *args)
functools.lru_cache(maxsize=128)

# IMPORT PATTERNS
import module
from module import name
from module import name as alias
from package import module

# STANDARD LIBRARY HIGHLIGHTS
math  random  datetime  os  sys  re
collections  itertools  functools  pathlib
json  csv  typing  abc  dataclasses
```
